# 03 Student Overfit — 8-object

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import matplotlib.pyplot as plt
from meshuv.data.dataset import CleanDataset
from meshuv.visualization import pipeline_views as V

DATASET = "processed/clean_v1"      # 相对 MESHUV_DATA_ROOT(env 可覆盖)
root = V.resolve(DATASET)
ds = CleanDataset(root, expose_diagnostics=True) if root else None
print("objects:", ds and len(ds))

In [ ]:
import json, numpy as np, torch
from meshuv.data.collate import collate
from meshuv.model.student_v0 import StudentV0
from meshuv.training.trainer import evaluate
ck = torch.load("../reports/student_v0_sanity.ckpt", map_location="cpu")
model = StudentV0(d=ck["d"]); model.load_state_dict(ck["state"]); model.eval()
print("loaded ckpt:", {k: ck[k] for k in ("model", "seed", "lr", "steps", "device")})
ids = ck["uids"]["train"][:8]
items = [ds[ds.ids.index(o)] for o in ids if o in ds.ids]
batch = collate(items)
ev = evaluate(model, items, collate)
with torch.no_grad():
    pred = model(torch.as_tensor(batch["features"]), batch["object_ranges"],
                 torch.as_tensor(batch["valid"])).numpy()
r = dict(pred=pred, losses=[])
print(json.dumps(ev["macro"], indent=1))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
m = batch["valid"]
ax.scatter(batch["target"][m], r["pred"][m], s=8, alpha=0.6)
lim = float(np.abs(batch["target"][m]).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], "r--", lw=1)
ax.set_xlabel("target"); ax.set_ylabel("pred(ckpt)")
plt.tight_layout(); plt.show()

In [ ]:
OBJ = 0
item = items[OBJ]
a, b = batch["object_ranges"][OBJ]
vmax = float(np.abs(batch["target"][a:b]).max())
fig = plt.figure(figsize=(13, 4.5))
V.show_target(item, fig.add_subplot(1, 3, 1), vmax=vmax)
V.show_prediction(item, r["pred"][a:b], fig.add_subplot(1, 3, 2), vmax=vmax)
V.show_charts(item, fig.add_subplot(1, 3, 3))
plt.tight_layout(); plt.show()